# WienerNet — Manuscript Plots

Reproduces the figures from the original `All_Seed_Experiments_Night.ipynb`
plus two new plots (predicted-vs-GT noise distribution, overall and per site).

The plotting logic lives in `wienernet.evaluation.plots`. This notebook is the
**orchestration layer** that:
1. Loads site data (for data-exploration plots)
2. Loads training runs (for per-model plots)
3. Calls the plotting functions and saves outputs to `notebooks/manuscript_plots/`

Pre-requisite: at least one training run per variant must exist in `outputs/`.
Produce one with e.g. `python scripts/train.py model=piae_sde_sampling`.


In [ ]:
# Make `wienernet` importable when running from the notebooks/ dir.
import sys
from pathlib import Path
_repo_root = Path.cwd().resolve()
while _repo_root != _repo_root.parent and not (_repo_root / 'wienernet').is_dir():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from wienernet.data import load_site_parquets, build_dataloaders, physics_nee_numpy
from wienernet.evaluation import (
    setup_paper_style,
    plot_sample_counts_per_site,
    plot_nee_distribution_grid,
    plot_noise_distribution_grid,
    plot_feature_importance,
    plot_all_temporal_scales,
    plot_nee_daily, plot_nee_weekly, plot_nee_monthly, plot_nee_quarterly,
    plot_noise_comparison_overall, plot_noise_comparison_per_site,
    compute_gt_noise,
)
from wienernet.models import build_model
from wienernet.utils import load_model_weights, set_seed_globally
from wienernet.training import Trainer
import torch

logging.basicConfig(level=logging.WARNING, format="%(name)s: %(message)s")
setup_paper_style()


## 1. Configuration

Edit the `SITE_PATHS` and `RUN_DIRS` dictionaries below to point at your data
and trained runs. Set `RUN_DIRS[variant] = None` to skip that variant.


In [ ]:
REPO_ROOT = Path("..").resolve()
OUT_DIR = REPO_ROOT / "notebooks" / "manuscript_plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Site data paths (mirror configs/data/east_anglia.yaml; add wicken_fen + woodwalton
# if you've trained with all 6 sites)
SITE_PATHS = {
    "Redmere1":  REPO_ROOT / "data_manipulation/other_sites/redmere_1/final_night_data.parquet",
    "Redmere2":  REPO_ROOT / "data_manipulation/other_sites/redmere_2/final_night_data.parquet",
    "Great Fen": REPO_ROOT / "data_manipulation/other_sites/great_fen/final_night_data.parquet",
    "Rosedene":  REPO_ROOT / "data_manipulation/final_night_data.parquet",
    # 'Wicken Fen': REPO_ROOT / 'data_manipulation/other_sites/wicken_fen/final_night_data.parquet',
    # 'Woodwalton': REPO_ROOT / 'data_manipulation/other_sites/woodwalton/final_night_data.parquet',
}

# Mapping pretty-name -> internal site name used in build_dataloaders
SITE_NAME_TO_INTERNAL = {
    "Redmere1": "redmere_1", "Redmere2": "redmere_2",
    "Great Fen": "great_fen", "Rosedene": "rosedene",
    "Wicken Fen": "wicken_fen", "Woodwalton": "woodwalton",
}

# Auto-discover the latest run per variant. Override with explicit paths below
# if you want a specific seed / experiment.
def _latest_run(variant: str) -> Path | None:
    matches = sorted(REPO_ROOT.glob(f"outputs/*_{variant}_seed*"))
    return matches[-1] if matches else None

RUN_DIRS = {
    "PIAE":     _latest_run("piae_sde_sampling"),
    "PIAE_Reg": _latest_run("piae_sde_reg_sampling"),
    "PIVAE":    _latest_run("pivae_sde_sampling"),
    "AE":       _latest_run("ae"),
    "VAE":      _latest_run("vae"),
    "RF":       _latest_run("rf"),
    "XGB":      _latest_run("xgb"),
}
for name, path in RUN_DIRS.items():
    print(f"  {name:9s} -> {path if path else 'NOT FOUND'}")


## 2. Data-exploration plots (no model required)

In [ ]:
# Load each site's final_night_data.parquet
site_dfs = {}
for pretty, path in SITE_PATHS.items():
    if not path.exists():
        print(f"  [skip] {pretty}: {path} not found"); continue
    df = pd.read_parquet(path)
    df["NEE_phy"] = physics_nee_numpy(df["E0"].values, df["rb"].values, df["Ta"].values)
    site_dfs[pretty] = df
    print(f"  {pretty:10s}  rows={len(df):>6,d}")


In [ ]:
# Plot 1: Sample counts per site
counts = {name: len(df) for name, df in site_dfs.items()}
fig = plot_sample_counts_per_site(counts, save_path=OUT_DIR / "01_sample_counts_per_site.png")
plt.show()


In [ ]:
# Plot 2: NEE vs NEE_phy distribution per site
fig = plot_nee_distribution_grid(site_dfs, save_path=OUT_DIR / "02_nee_vs_neephy_per_site.png")
plt.show()


In [ ]:
# Plot 3: Ground-truth noise (NEE_phy - NEE) per site
fig = plot_noise_distribution_grid(site_dfs, save_path=OUT_DIR / "03_noise_distribution_per_site.png")
plt.show()


## 3. Load predictions from each trained run

For torch models, predictions live at `<run>/metrics/predictions.parquet`
(produced by `scripts/evaluate.py`). For baselines, they live at
`<run>/predictions.parquet`. If you haven't evaluated a torch run yet, the
cell below will run inference on the fly.


In [ ]:
WIENERNET_VARIANTS = {"piae_sde_sampling", "piae_sde_reg_sampling", "pivae_sde_sampling"}

def _load_predictions(run_dir: Path) -> pd.DataFrame:
    """Return a dataframe with columns gt_*, pred_*, plus the test_data columns."""
    # Baseline: predictions.parquet sits in the run dir
    candidate_baseline = run_dir / "predictions.parquet"
    candidate_torch    = run_dir / "metrics" / "predictions.parquet"
    if candidate_baseline.exists():
        return pd.read_parquet(candidate_baseline)
    if candidate_torch.exists():
        return pd.read_parquet(candidate_torch)
    raise FileNotFoundError(
        f"No predictions found in {run_dir}. "
        f"For torch runs, run: python scripts/evaluate.py --run {run_dir}"
    )


loaded: dict[str, dict] = {}   # name -> {df, gt, preds, test_data}
for name, run_dir in RUN_DIRS.items():
    if run_dir is None: continue
    try:
        df = _load_predictions(run_dir)
    except FileNotFoundError as e:
        print(f"  [skip] {name}: {e}"); continue

    gt = {col[len("gt_"):]: df[col].to_numpy() for col in df.columns if col.startswith("gt_")}
    preds = {col[len("pred_"):]: df[col].to_numpy() for col in df.columns if col.startswith("pred_")}
    test_data = df.drop(columns=[c for c in df.columns if c.startswith(("gt_", "pred_"))])
    loaded[name] = dict(df=df, gt=gt, preds=preds, test_data=test_data, run_dir=run_dir)
    print(f"  {name:9s}  {len(df):>6,d} rows  pred keys: {sorted(preds)}")


## 4. Temporal-scale plots (daily / weekly / monthly / quarterly)

Each model gets 4 panels at the standard scales. The PIAE variant additionally
shows predictions-without-noise, the physics line, and a 95% prediction interval.


In [ ]:
def _temporal_kwargs_for(name: str) -> dict:
    """PIAE_SDE_Sampling gets the rich daily panel; others stay simple."""
    if name == "PIAE":
        return {
            "daily_kwargs": dict(
                show_predictions_no_noise=True,
                show_physics=True,
                show_ci=True,
            ),
            "weekly_kwargs": dict(show_residual_ci=True),
        }
    return {}


for name, entry in loaded.items():
    # 'test_data' needs the season column for the quarterly plot — recompute if missing
    td = entry["test_data"]
    if "season" not in td.columns:
        # Map month -> NH season (winter=0, spring=1, summer=2, autumn=3)
        from wienernet.data import set_season_tag
        td = set_season_tag(td)
    if "NEE_phy" not in td.columns and {"E0", "rb", "Ta"}.issubset(td.columns):
        td = td.copy()
        td["NEE_phy"] = physics_nee_numpy(td["E0"].values, td["rb"].values, td["Ta"].values)
    entry["test_data"] = td

    figs = plot_all_temporal_scales(
        entry["gt"], entry["preds"], td,
        model_name=name,
        save_dir=OUT_DIR / "temporal" / name,
        save_prefix=name.lower(),
        **_temporal_kwargs_for(name),
    )
    print(f"  {name}: {list(figs)}")
plt.show()


## 5. Predicted noise vs ground-truth noise (WienerNet variants only)

The ground-truth noise is `NEE - NEE_phy` (Lloyd-Taylor residual). The model's
noise head predicts the noise sample, and `noise_stds` predicts the per-sample
standard deviation. For the model to be physically faithful, the two histograms
should overlap.


In [ ]:
# Map pretty name -> internal variant key, used to identify WienerNet runs
PRETTY_TO_VARIANT = {
    "PIAE": "piae_sde_sampling",
    "PIAE_Reg": "piae_sde_reg_sampling",
    "PIVAE": "pivae_sde_sampling",
}

for name, entry in loaded.items():
    variant = PRETTY_TO_VARIANT.get(name)
    if variant not in WIENERNET_VARIANTS:
        continue
    if "noise" not in entry["preds"]:
        print(f"  [skip] {name}: no 'noise' predictions saved"); continue
    td = entry["test_data"]
    # Make sure GT noise can be computed
    required = {"NEE", "E0", "rb", "Ta"}
    if not required.issubset(td.columns):
        missing = required - set(td.columns)
        print(f"  [skip] {name}: missing columns {missing}"); continue

    gt_noise = compute_gt_noise(td)
    pred_noise = entry["preds"]["noise"]

    # Overall
    fig = plot_noise_comparison_overall(
        gt_noise, pred_noise,
        model_name=name,
        save_path=OUT_DIR / f"05_noise_overall_{name.lower()}.png",
    )
    plt.show()

    # Per site
    if "site" in td.columns:
        fig = plot_noise_comparison_per_site(
            td, gt_noise, pred_noise,
            model_name=name,
            save_path=OUT_DIR / f"05_noise_per_site_{name.lower()}.png",
        )
        plt.show()
    else:
        print(f"  [skip per-site] {name}: no 'site' column in test_data")


## 6. Baseline feature importance (RF / XGB)

In [ ]:
import joblib

for name in ("RF", "XGB"):
    run_dir = RUN_DIRS.get(name)
    if run_dir is None: continue
    csv_path = run_dir / "feature_importance.csv"
    if not csv_path.exists():
        print(f"  [skip] {name}: feature_importance.csv missing"); continue
    fi = pd.read_csv(csv_path)
    fig = plot_feature_importance(
        fi["feature"].tolist(), fi["importance"].to_numpy(),
        top_n=15,
        title=f"{name}: Feature Importances",
        save_path=OUT_DIR / f"06_feature_importance_{name.lower()}.png",
    )
    plt.show()


## 7. Output summary

All figures are saved under `notebooks/manuscript_plots/`. Sub-directories:
- `temporal/<MODEL>/*.png` — daily / weekly / monthly / quarterly per model
- Top-level: data-exploration + noise-comparison + feature-importance plots


In [ ]:
# List everything we produced
for path in sorted(OUT_DIR.rglob("*.png")):
    print(f"  {path.relative_to(REPO_ROOT)}")
